In [ ]:
import h5py
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- 1. LOAD EMBEDDINGS FROM HDF5 ---
H5_PATH = "../embeddings/task4_dna_rna_Vir2vec-422M.h5"

with h5py.File(H5_PATH, "r") as f:
    X = np.array(f["embeddings"][:])
    raw_labels = [l.decode("utf-8") if isinstance(l, bytes) else str(l) for l in f["labels"][:]]

encoder = LabelEncoder()
y = encoder.fit_transform(raw_labels)
class_names = encoder.classes_.tolist()

print(f"Loaded embeddings: X shape={X.shape}, Classes={class_names}")

# --- 2. PIPELINE & HYPERPARAMETER GRID ---
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42))
])

param_grid = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__class_weight": [None, "balanced"]
}

# --- 3. NESTED 5x3 STRATIFIED CROSS-VALIDATION ---
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Inner CV for Hyperparameter Tuning
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42 + fold_idx)
    grid = GridSearchCV(pipeline, param_grid, scoring="balanced_accuracy", cv=inner_cv, n_jobs=-1)
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    # Compute Outer Test Fold Metrics
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    fold_metrics.append({
        "fold": fold_idx,
        "balanced_accuracy": bal_acc,
        "accuracy": acc,
        "macro_f1": f1,
        "mcc": mcc,
        "best_params": grid.best_params_
    })
    print(f"Fold {fold_idx}: Balanced Acc = {bal_acc:.4f} | Best C = {grid.best_params_['clf__C']}")

# --- 4. SUMMARY RESULTS ---
df_res = pd.DataFrame(fold_metrics)
print("\n--- Final Logistic Regression Results ---")
print(f"Macro Balanced Accuracy: {df_res['balanced_accuracy'].mean():.4f} ± {df_res['balanced_accuracy'].std():.4f}")
print(f"Macro F1 Score:        {df_res['macro_f1'].mean():.4f} ± {df_res['macro_f1'].std():.4f}")